# F4·F5 EarnMore 시험 (notes/funnel_protocol.md)
런타임: **GPU**. 위에서부터 차례로 실행합니다. 결과는 Drive `data/funnel/earnmore/`에 저장됩니다.

- 공식 코드 DVampire/EarnMore @810ff59, 수정 없음.
- **성과 비노출:** 학습 2007-09-26~2014-01-01, 검증 2014-01-01~2015-01-01. 논문의 학습 구간 안이므로 시험 구간을 쓰지 않습니다. 출력은 파일로만 보내고 화면에는 설치 결과와 시간만 냅니다.
- 2 에피소드만 돌려 시간을 잽니다. 이 노트북으로 재현 판정을 하지 않습니다.
- 1차 시도(11-05): `make_scripts.py`가 yapf 신버전과 충돌(`FormatCode(verify=)`), `train.py`가 requirements.txt에 없는 plotly에서 실패. 환경만 보강: yapf 0.40.1, plotly·statsmodels 추가, 설치 직후 `train.py --help`로 import 전체 확인.
- qlib(README 안내, master 미고정)은 휠 빌드 실패 시 건너뜀: 공식 코드 어디서도 import하지 않음.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib, time, json
OUT = pathlib.Path('/content/drive/MyDrive/data/funnel/earnmore'); OUT.mkdir(parents=True, exist_ok=True)
START = time.time(); (OUT/'log.txt').open('a').write(f'start {time.ctime()}\n')
!nvidia-smi -L

In [ ]:
%%bash
set -e
cd /content
curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
./bin/micromamba create -y -q -p /content/em -c conda-forge python=3.9 pip 'setuptools<66' 'wheel<0.40'
[ -d EarnMore ] || git clone -q https://github.com/DVampire/EarnMore.git
cd EarnMore && git checkout -q 810ff59 && git log -1 --format='%h %ad' --date=short

In [ ]:
%%bash
# 설치(공식 코드는 수정하지 않음). 각 단계 로그는 /content/out/install_*.log, 실패하면 그 단계에서 멈추고 로그 끝을 보여 준다.
mkdir -p /content/out; cd /content
P="/content/em/bin/python -m pip"
D=/content/drive/MyDrive/data/funnel/earnmore
step(){ n=$1; shift; echo "== $n"; if ! "$@" > /content/out/install_$n.log 2>&1; then echo "실패: $n"; tail -25 /content/out/install_$n.log; cp /content/out/install_*.log "$D"/ 2>/dev/null; echo "로그를 Drive에 저장함"; exit 1; fi; tail -1 /content/out/install_$n.log; }
step tools $P install -q "pip<24.1" "setuptools==65.5.0" "wheel==0.38.4"
step torch $P install -q torch==1.13.1+cu117 --extra-index-url https://download.pytorch.org/whl/cu117
step gym $P install -q --no-build-isolation gym==0.21.0
[ -d qlib ] || git clone -q https://github.com/microsoft/qlib.git
# qlib: README 설치 안내에만 있고 공식 코드(pm/·tools/·configs/)는 import하지 않음 → 시도하되 실패해도 계속(아래 importcheck가 실제 관문)
$P install -q "cython<3" "numpy<1.24" > /content/out/install_qlibdeps.log 2>&1
if ! bash -c "cd qlib && $P install -q --no-build-isolation ." > /content/out/install_qlib.log 2>&1; then echo "== qlib 설치 실패(공식 코드 미사용, 계속 진행)"; grep -iE "error" /content/out/install_qlib.log | head -5; else echo "== qlib ok"; fi
step reqs bash -c "cd EarnMore && $P install -q --no-build-isolation -r requirements.txt"
# requirements.txt에 없지만 공식 코드가 import하는 것(pm/utils/plot.py: plotly, statsmodels), mmengine 0.7.2와 맞는 yapf
step extras $P install -q "yapf==0.40.1" "plotly==5.14.1" "statsmodels==0.13.5" "pandas==1.2.4" "scipy==1.8.0"
# 공식 train.py의 import 전체 확인(--help는 import 후 인자만 해석하고 끝남)
step importcheck bash -c "cd EarnMore && PYTHONPATH=. /content/em/bin/python tools/train.py --help"
/content/em/bin/python - <<'PY'
import importlib
for m in ["torch","gym","pandas","numpy","mmengine","timm","einops","qlib","yapf","plotly","statsmodels"]:
    try:
        x=importlib.import_module(m); print(m, getattr(x,"__version__","?"))
    except Exception as e:
        print("IMPORT 실패", m, type(e).__name__, str(e)[:200])
import torch; print("cuda", torch.cuda.is_available())
PY
cp /content/out/install_*.log "$D"/ 2>/dev/null; true


In [ ]:
%%bash
cd /content/EarnMore
mkdir -p /content/out
export PYTHONPATH=$PWD
/content/em/bin/python tools/preprocess.py > /content/out/preprocess.log 2>&1; echo preprocess exit $?
ls datasets/dj30/features | wc -l
/content/em/bin/python tools/make_scripts.py --config configs/mask_sac_portfolio_management.py --mask --action_wrapper_method softmax --num_episodes 2000 --dataset dj30 --num_stocks 28 --buffer_size 10000 --repeat_times 128 --gpu_id 0 --days 10 --lr 1e-5 --act_lr 1e-5 --cri_lr 1e-5 --rep_lr 1e-5 --beta_lr 1e-5 --seed 42 --T 0.1 --data_group 0 > /content/out/make_scripts.log 2>&1; rc=$?; echo make_scripts exit $rc; [ $rc -eq 0 ] || { tail -5 /content/out/make_scripts.log; exit 1; }
git status --short | grep -v features | head

위 칸의 마지막 줄에 나온 생성 파일을 다음 칸이 자동으로 찾습니다. 오류가 나면 그대로 알려 주세요.

In [ ]:
import glob, subprocess, shutil, re
os.chdir('/content/EarnMore')
new = subprocess.check_output('git status --short --untracked-files=all', shell=True, text=True).split('\n')
cfgs = [l[3:] for l in new if l[3:].endswith('.py') and 'configs' in l]
print(cfgs[:5])
assert cfgs and os.path.getsize(cfgs[0]) > 0, '생성된 설정 파일 없음/빈 파일'
cfg = cfgs[0]
env = dict(os.environ, PYTHONPATH=os.getcwd())
opts = ['num_episodes=2','train_start_date=2007-09-26','val_start_date=2014-01-01','test_start_date=2015-01-01','test_end_date=2015-06-01']
t0 = time.time()
with open('/content/out/smoke.log','w') as f:
    rc = subprocess.call(['/content/em/bin/python','tools/train.py','--config',cfg,'--workdir','/content/out/work','--cfg-options',*opts], stdout=f, stderr=f, env=env)
sec = time.time()-t0
txt = open('/content/out/smoke.log', errors='ignore').read()
eps = len(re.findall(r'Train Episode: \[', txt))
ok = (rc == 0 and eps >= 1)  # 실패·0 에피소드면 시간 추정을 내지 않음
r = {'rc':rc,'seconds':sec,'episodes':eps,'hours_per_seed_at_2000':(sec/eps*2000/3600 if ok else None),'gpu':subprocess.getoutput('nvidia-smi -L'),'config':cfg}
print(json.dumps(r, indent=1))
(OUT/'f4_timing.json').write_text(json.dumps(r, indent=1))

In [ ]:
# 오류가 나면 마지막 줄만 보여 줍니다(성과 표는 출력하지 않음).
import subprocess
for n in ['preprocess.log','make_scripts.log','smoke.log']:
    shutil.copy('/content/out/'+n, OUT/n)
print(subprocess.getoutput("grep -nE 'Error|Traceback|error:' /content/out/smoke.log | tail -5"))
print('작업 시간(시간):', round((time.time()-START)/3600,2))
(OUT/'log.txt').open('a').write(f'end {time.ctime()}\n')